In [1]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")

NHK_ROOT = Path("/content/drive/MyDrive/CS313_Nhom5")

assert NHK_ROOT.is_dir(), (
    "Chưa thấy CS313_Nhom5. Kiểm tra quyền chia sẻ và lối tắt "
    "trong Drive của tôi của tài khoản mới."
)

required_data = [
    "PSM/train.csv",
    "PSM/test.csv",
    "PSM/test_label.csv",
    "MSL/MSL_train.npy",
    "MSL/MSL_test.npy",
    "MSL/MSL_test_label.npy",
    "SMD/SMD_train.npy",
    "SMD/SMD_test.npy",
    "SMD/SMD_test_label.npy",
]

missing = [
    name for name in required_data
    if not (NHK_ROOT / "data" / name).is_file()
]

assert not missing, f"Chưa truy cập được các file: {missing}"
assert (NHK_ROOT / "hoang_khang/E3a").is_dir(), (
    "Chưa thấy thư mục kết quả cũ."
)
assert torch.cuda.is_available(), (
    "Tài khoản mới chưa được cấp GPU. Chưa bắt đầu train."
)

print("ĐÃ THẤY DỮ LIỆU VÀ KẾT QUẢ CŨ.")
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
ĐÃ THẤY DỮ LIỆU VÀ KẾT QUẢ CŨ.
GPU: Tesla T4


In [2]:
import ast
import json
import random
import hashlib

import joblib
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader, Subset
from google.colab import files

print("Chọn file NHK_code (9).ipynb trên máy bạn.")
uploaded = files.upload()

assert len(uploaded) == 1, "Chỉ chọn một notebook."
filename, content = next(iter(uploaded.items()))
assert filename.endswith(".ipynb"), "Cần chọn notebook .ipynb."

backup = json.loads(content.decode("utf-8-sig"))

wanted_constants = {
    "PSM_EXPECTED_SHA",
    "EXTRA_DATA_REFERENCES",
}
wanted_definitions = {
    "PSMWindowDataset",
    "load_usad_shared_data",
}

restored_constants = {}
restored_nodes = {}

for cell in backup["cells"]:
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if (
                    isinstance(target, ast.Name)
                    and target.id in wanted_constants
                ):
                    restored_constants[target.id] = ast.literal_eval(
                        node.value
                    )

        elif isinstance(node, (ast.ClassDef, ast.FunctionDef)):
            if node.name in wanted_definitions:
                restored_nodes[node.name] = node

assert wanted_constants.issubset(restored_constants), (
    "Notebook thiếu metadata dữ liệu."
)
assert wanted_definitions.issubset(restored_nodes), (
    "Notebook thiếu Dataset hoặc helper dữ liệu chung."
)

globals().update(restored_constants)

# Chỉ nạp hai định nghĩa; KHÔNG chạy bất kỳ ô train cũ nào.
for name in ["PSMWindowDataset", "load_usad_shared_data"]:
    module = ast.Module(
        body=[restored_nodes[name]],
        type_ignores=[]
    )
    exec(
        compile(module, f"restore_{name}", "exec"),
        globals()
    )

print("KHÔI PHỤC XONG PHẦN DỮ LIỆU CHUNG.")
print("Chưa tải lại data và chưa train lại lượt nào.")

Chọn file NHK_code (9).ipynb trên máy bạn.


Saving NHK_code (9).ipynb to NHK_code (9).ipynb
KHÔI PHỤC XONG PHẦN DỮ LIỆU CHUNG.
Chưa tải lại data và chưa train lại lượt nào.


In [3]:
from pathlib import Path
import hashlib

expected_sha = (
    "ea118dffa00eb53f6439f109cf7f9271"
    "69ce7b6f56e9136c43aa161552c1f323"
)

GANF_ADAPTER_PATH = (
    NHK_ROOT
    / "source_cache"
    / ("GANF_adapter_" + expected_sha[:12])
    / "ganf_colab_cells.py"
)

assert GANF_ADAPTER_PATH.is_file(), (
    f"Chưa thấy file GANF đã lưu: {GANF_ADAPTER_PATH}. "
    "Dừng và gửi thông báo này, không chạy train."
)

ganf_content = GANF_ADAPTER_PATH.read_bytes()

assert hashlib.sha256(ganf_content).hexdigest() == expected_sha, (
    "File GANF không đúng bản đã kiểm thử."
)

exec(
    compile(
        ganf_content.decode("utf-8-sig"),
        str(GANF_ADAPTER_PATH),
        "exec"
    ),
    globals()
)

print("SẴN SÀNG CHẠY GANF TRÊN TÀI KHOẢN MỚI.")

GANF đã nạp, CHƯA train. Dùng dữ liệu/scaler/split chung; checkpoint lưu cả model+A.
SẴN SÀNG CHẠY GANF TRÊN TÀI KHOẢN MỚI.


In [5]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [8]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import torch
import gc

# Thư mục dùng chung: giữ nguyên dữ liệu và 43 lượt đã hoàn tất.
NHK_SHARED_E3A = Path(
    "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"
)

# Thư mục MỚI, tạo trực tiếp trong Drive của tài khoản đang mount.
# Không đặt bên trong lối tắt CS313_Nhom5.
NHK_OWN_E3A = Path(
    "/content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a"
)

assert NHK_SHARED_E3A.is_dir(), "Không thấy kết quả cũ."
assert torch.cuda.is_available(), "Không còn GPU."

NHK_OWN_E3A.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
probe = NHK_OWN_E3A / f"WRITE_CHECK_{stamp}"
probe.mkdir(exist_ok=False)

pd.DataFrame({"check": [1]}).to_csv(
    probe / "check.csv", index=False
)
assert pd.read_csv(probe / "check.csv")["check"].tolist() == [1]

# Chỉ đổi root OUTPUT của GANF.
# Helper dữ liệu vẫn đọc data/scaler/split từ CS313_Nhom5 cũ.
GANF_ROOT = NHK_OWN_E3A

gc.collect()
torch.cuda.empty_cache()

print("KIỂM TRA GHI THÀNH CÔNG.")
print("GANF sẽ lưu kết quả mới tại:", GANF_ROOT)
print("Dữ liệu/scaler/split cũ không thay đổi.")

KIỂM TRA GHI THÀNH CÔNG.
GANF sẽ lưu kết quả mới tại: /content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a
Dữ liệu/scaler/split cũ không thay đổi.


In [9]:
GANF_SMD17_RUN = run_ganf_if_missing("SMD", 17)

GANF SMD seed17: train/val/test=56668/14167/70837
Lịch train nguồn: augmented DAG rounds +30 final epochs; batch512. Output: /content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a/GANF_SMD_seed17_20261123T054100_285643Z
GANF SMD seed17: epoch 001 [augmented] NLL=-1.724095 h=0.0641098 rho=1
GANF SMD seed17: epoch 002 [augmented] NLL=-0.710614 h=0.0335007 rho=1
GANF SMD seed17: epoch 003 [augmented] NLL=-2.250609 h=0.0102119 rho=10
GANF SMD seed17: epoch 004 [augmented] NLL=6.051184 h=0.00473022 rho=10
GANF SMD seed17: epoch 005 [augmented] NLL=-1.363024 h=0.00395584 rho=10
GANF SMD seed17: epoch 006 [augmented] NLL=7.679322 h=0.00249481 rho=100
GANF SMD seed17: epoch 007 [augmented] NLL=-2.627711 h=0.000205994 rho=1e+03
GANF SMD seed17: epoch 008 [augmented] NLL=-0.608239 h=0.00030899 rho=1e+03
GANF SMD seed17: epoch 009 [augmented] NLL=0.901876 h=0.000106812 rho=1e+04
GANF SMD seed17: epoch 010 [augmented] NLL=-2.911862 h=1.14441e-05 rho=1e+05
GANF SMD seed17: epoch 011 [final] NLL=7.639369 h

In [10]:
GANF_SMD18_RUN = run_ganf_if_missing("SMD", 18)

GANF SMD seed18: train/val/test=56668/14167/70837
Lịch train nguồn: augmented DAG rounds +30 final epochs; batch512. Output: /content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a/GANF_SMD_seed18_20261123T055945_723524Z
GANF SMD seed18: epoch 001 [augmented] NLL=-1.371197 h=0.0728149 rho=1
GANF SMD seed18: epoch 002 [augmented] NLL=-1.766376 h=0.0296402 rho=1
GANF SMD seed18: epoch 003 [augmented] NLL=-1.655004 h=0.00688171 rho=1
GANF SMD seed18: epoch 004 [augmented] NLL=-1.794950 h=0.00205994 rho=1
GANF SMD seed18: epoch 005 [augmented] NLL=-1.632554 h=0.000549316 rho=1
GANF SMD seed18: epoch 006 [augmented] NLL=-2.149113 h=0.000289917 rho=1
GANF SMD seed18: epoch 007 [augmented] NLL=-0.071759 h=0.000656128 rho=10
GANF SMD seed18: epoch 008 [augmented] NLL=-1.540336 h=9.15527e-05 rho=100
GANF SMD seed18: epoch 009 [final] NLL=-2.222226 h=4.57764e-05 rho=100
GANF SMD seed18: epoch 010 [final] NLL=-3.250526 h=8.39233e-05 rho=100
GANF SMD seed18: epoch 011 [final] NLL=-3.306682 h=0.000236511 r

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

paths = [
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/CS313_Nhom5",
    "/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a",
    "/content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a",
]

for text in paths:
    print(Path(text).is_dir(), "|", text)

Mounted at /content/drive
True | /content/drive/MyDrive
True | /content/drive/MyDrive/CS313_Nhom5
True | /content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a
True | /content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a


In [4]:
import ast
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

project = Path("/content/drive/MyDrive/CS313_Nhom5")
expected_sha = (
    "ea118dffa00eb53f6439f109cf7f9271"
    "69ce7b6f56e9136c43aa161552c1f323"
)

GANF_ADAPTER_PATH = (
    project / "source_cache"
    / ("GANF_adapter_" + expected_sha[:12])
    / "ganf_colab_cells.py"
)

assert GANF_ADAPTER_PATH.is_file(), (
    f"Không thấy file đã lưu: {GANF_ADAPTER_PATH}"
)

content = GANF_ADAPTER_PATH.read_bytes()
assert hashlib.sha256(content).hexdigest() == expected_sha

tree = ast.parse(content.decode("utf-8-sig"))

# Chỉ đọc metadata nguồn, không chạy setup/train cũ.
metadata = {}

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if (
                isinstance(target, ast.Name)
                and target.id in {"GANF_COMMIT", "GANF_HASHES"}
            ):
                metadata[target.id] = ast.literal_eval(node.value)

assert set(metadata) == {"GANF_COMMIT", "GANF_HASHES"}
globals().update(metadata)

GANF_CACHE = project / "source_cache" / ("GANF_" + GANF_COMMIT)

for name in ("models/GANF.py", "models/NF.py"):
    path = GANF_CACHE / name
    assert path.is_file(), f"Thiếu source cache: {path}"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == GANF_HASHES[name]

# Tạo lại fingerprint source; KHÔNG nạp hoặc tạo model.
network_tree = ast.parse(
    (GANF_CACHE / "models/GANF.py").read_text(encoding="utf-8")
)
network_tree.body = [
    node for node in network_tree.body
    if not (
        isinstance(node, ast.ImportFrom)
        and node.module == "models.NF"
    )
]

GANF_NETWORK_CODE = (
    (GANF_CACHE / "models/NF.py").read_text(encoding="utf-8")
    + "\n\n" + ast.unparse(network_tree) + "\n"
)

functions = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name in {"ganf_spec", "export_nhk_handover"}
]

assert {node.name for node in functions} == {
    "ganf_spec", "export_nhk_handover"
}

exec(
    compile(
        ast.Module(body=functions, type_ignores=[]),
        "restore_export_only",
        "exec"
    ),
    globals()
)

GANF_ROOT = Path(
    "/content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a"
)

print("ĐÃ KHÔI PHỤC PHẦN XUẤT BÁO CÁO. Chưa train lại gì.")

ĐÃ KHÔI PHỤC PHẦN XUẤT BÁO CÁO. Chưa train lại gì.


In [6]:
from datetime import datetime, timezone
import shutil

In [7]:
import ast
import hashlib
import json
from pathlib import Path

NHK_HANDOVER_ROOTS = [
    Path("/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a"),
    Path("/content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a"),
]

for folder in NHK_HANDOVER_ROOTS:
    assert folder.is_dir(), f"Chưa truy cập được: {folder}"

assert "GANF_ADAPTER_PATH" in globals() and "ganf_spec" in globals(), (
    "Thiếu định nghĩa GANF. Chỉ nạp setup, không chạy lại train."
)

def _nhk_find_runs(pattern):
    return [
        folder
        for root in NHK_HANDOVER_ROOTS
        for folder in root.glob(pattern)
    ]

# Dùng lại toàn bộ kiểm tra của exporter đã có,
# chỉ đổi sang đọc cả hai thư mục.
tree = ast.parse(
    Path(GANF_ADAPTER_PATH).read_text(encoding="utf-8-sig")
)
node = next(
    n for n in tree.body
    if isinstance(n, ast.FunctionDef)
    and n.name == "export_nhk_handover"
)

export_code = ast.unparse(node)
assert export_code.count("GANF_ROOT.glob(") == 1
assert export_code.count("reverse=True") == 1

export_code = export_code.replace(
    "GANF_ROOT.glob(", "_nhk_find_runs("
).replace(
    "reverse=True", "key=lambda p: p.name, reverse=True"
)

exec(
    compile(export_code, "nhk_multi_drive_export_only", "exec"),
    globals()
)

# Ghi báo cáo vào Drive riêng của tài khoản2.
GANF_ROOT = NHK_HANDOVER_ROOTS[1]
HANDOVER_DIR, ALL_RESULTS, ALL_SUMMARY = export_nhk_handover()

readme = HANDOVER_DIR / "README_BAN_GIAO.md"
with readme.open("a", encoding="utf-8") as handle:
    handle.write(
        "\n## Kết quả nằm trên HAI tài khoản\n"
        "- Giữ/share CS313_Nhom5 của tài khoản1.\n"
        "- Giữ/share NHK_GANF_TAI_KHOAN_2/E3a của tài khoản2.\n"
        "- run_inventory.csv chỉ tới artifact gốc, không sao chép model.\n"
    )

checksums = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in HANDOVER_DIR.iterdir()
    if p.is_file() and p.name != "report_checksums.json"
}
(HANDOVER_DIR / "report_checksums.json").write_text(
    json.dumps(checksums, indent=2), encoding="utf-8"
)

display(ALL_SUMMARY)
print("BÀN GIAO45 LƯỢT TỪ HAI NƠI:", HANDOVER_DIR)

ĐÃ KIỂM TRA ĐỦ45 LƯỢT VÀ XUẤT BÁO CÁO: /content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a/HANDOVER_NHK_20261123T072905_496378Z
Còn bước người dùng: lưu notebook và share các thư mục artifact gốc trên Drive.


,model,variant,dataset,n_seeds,auroc_mean,auroc_std,auprc_mean,auprc_std,runtime_s_mean,epoch_min,epoch_max
0,ALOCC,mtgflow_cnn_width16_safe_probe,MSL,3,0.507465,0.044002,0.170076,0.027530,14.088809,40,40
1,ALOCC,mtgflow_cnn_width16_safe_probe,PSM,3,0.492288,0.072702,0.369786,0.085663,26.207391,40,40
2,ALOCC,mtgflow_cnn_width16_safe_probe,SMD,3,0.509665,0.040259,0.082696,0.001843,129.455272,40,40
3,DeepSVDD,mtgflow_repo_cnn_no_pretrain,MSL,3,0.540428,0.036925,0.186319,0.013259,4.850662,20,20
4,DeepSVDD,mtgflow_repo_cnn_no_pretrain,PSM,3,0.446208,0.057601,0.297876,0.030947,8.793438,20,20
5,DeepSVDD,mtgflow_repo_cnn_no_pretrain,SMD,3,0.752114,0.026049,0.197872,0.019936,41.408413,20,20
6,GANF,official_water_augLag_shared_standard,MSL,3,0.596790,0.011448,0.225927,0.004497,87.633752,43,45
7,GANF,official_water_augLag_shared_standard,PSM,3,0.689329,0.032072,0.541360,0.047307,87.265550,41,43
8,GANF,official_water_augLag_shared_standard,SMD,3,0.660570,0.016393,0.172080,0.023233,661.919831,38,43
9,MTGFlow,base,MSL,3,0.681755,0.032062,0.276125,0.026808,81.193214,40,40


BÀN GIAO45 LƯỢT TỪ HAI NƠI: /content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a/HANDOVER_NHK_20261123T072905_496378Z
